In [10]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import roc_auc_score

### first we need to train a SVM using whole image information

In [2]:
labels_path = '/host/d/Data/Habitats/Jishuitan/Patient_lists/labels_with_image_info_included_5fold.xlsx'
radiomics_path = '/host/d/projects/Habitats/radiomics/radiomics_measurements_svm_selected.xlsx'

labels_df = pd.read_excel(labels_path)
radiomics_df = pd.read_excel(radiomics_path)

non_feature_cols = ["Patient_index", "Image_filepath", "Mask_filepath"]
feature_cols = [c for c in radiomics_df.columns if c not in non_feature_cols]
X_all = radiomics_df[feature_cols].values

label_col = 'Pathologic_Response_Necrosis_gt90pct'
y_all = labels_df['Pathologic_Response_Necrosis_gt90pct'].values
fold_all = labels_df['fold'].values
print(f'Feature matrix shape: {X_all.shape}', f'Label vector shape: {y_all.shape}', f'Fold vector shape: {fold_all.shape}')

Feature matrix shape: (81, 13) Label vector shape: (81,) Fold vector shape: (81,)


In [ ]:


best_C = 100 #best_params["C"]  # best C = 100
best_gamma = 0.01 #best_params["gamma"] # best gamma: 0.01


df_preds = labels_df[['Patient_index', 'fold', 'Pathologic_Response_Necrosis_gt90pct']].copy()
df_preds["pred_prob"] = np.nan


# use all data as training and all data as validation

X_train, y_train = X_all, y_all
X_val,   y_val   = X_all, y_all

# 固定超参数训练模型
svm_fixed = SVC(
        kernel="rbf",
        C=best_C,
        gamma=best_gamma,
        probability=True,
        class_weight="balanced"
)
svm_fixed.fit(X_train, y_train)

# validation 预测概率（正类概率）
prob = svm_fixed.predict_proba(X_val)[:, 1]

# fold AUC
auc = roc_auc_score(y_val, prob)
print('AUC: {:.4f}'.format(auc))


AUC: 0.8881


In [20]:
### do it for each case - each patch

build = Build_list.Build(os.path.join('/host/d/Data/Habitats/Jishuitan/Patient_lists', 'labels_with_image_info_included.xlsx'))
batch_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')

for i in range(0, len(patient_index_list)):
    patient_index = patient_index_list[i]
    print('Processing patient index:', patient_index, ' i is ', i)

    # skip if done
    output_path = os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'radiomics_features_patches_prob_clusters.xlsx')
    # if os.path.isfile(output_path):
    #     print(f'Skipping patient index {patient_index} as output file already exists.')
    #     continue
    
    radiomics_path = os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'radiomics_features_patches_normalized_svm_selected.xlsx')
    radiomics_df = pd.read_excel(radiomics_path)


    non_feature_cols = ["Patient_index", "patch_id", "tumor_fraction",  "Image_filepath", "Mask_filepath"]
    feature_cols = [c for c in radiomics_df.columns if c not in non_feature_cols]
    X_all = radiomics_df[feature_cols].values

    # clip X_all values to [0,1]
    X_all = np.clip(X_all, 0, 1)

    # use svm_fixed to get the probability for each patch
    prob = svm_fixed.predict_proba(X_all)[:,1]
    print(f'Probability vector shape: {prob.shape}')


    # prob: shape (n_patches,)
    p = prob.reshape(-1, 1)   # KMeans 需要二维输入: (n_samples, n_features)

    K = 2
    kmeans = KMeans(n_clusters=K, random_state=0, n_init="auto")
    cluster_id = kmeans.fit_predict(p)   # shape (n_patches,)

    print("cluster_id shape:", cluster_id.shape)
    print("cluster counts:", np.bincount(cluster_id))

    # 每个cluster的均值prob（方便你理解哪个是高概率habitat）
    cluster_means = np.array([prob[cluster_id == c].mean() for c in range(K)])

    # make sure cluster means is higher in 1, if not, turn cluster 0 to 1 and cluster 1 to 0
    if cluster_means[0] > cluster_means[1]:
        cluster_id = 1 - cluster_id
    
    # check again
    cluster_means = np.array([prob[cluster_id == c].mean() for c in range(K)])
    print("cluster mean prob after adjustment:", cluster_means)

    # add prob to radiomics_df and save as a new file
    radiomics_df['prob'] = prob
    radiomics_df['cluster'] = cluster_id
    radiomics_df.to_excel(output_path, index=False)



Number of cases to process: 81
Processing patient index: 5  i is  0
Probability vector shape: (127,)
cluster_id shape: (127,)
cluster counts: [37 90]
cluster mean prob after adjustment: [0.54172136 0.91187935]
Processing patient index: 7  i is  1
Probability vector shape: (262,)
cluster_id shape: (262,)
cluster counts: [183  79]
cluster mean prob after adjustment: [0.61035756 0.93318896]
Processing patient index: 8  i is  2
Probability vector shape: (26,)
cluster_id shape: (26,)
cluster counts: [11 15]
cluster mean prob after adjustment: [0.31464795 0.77971809]
Processing patient index: 11  i is  3
Probability vector shape: (317,)
cluster_id shape: (317,)
cluster counts: [243  74]
cluster mean prob after adjustment: [0.57812255 0.92702946]
Processing patient index: 15  i is  4
Probability vector shape: (124,)
cluster_id shape: (124,)
cluster counts: [67 57]
cluster mean prob after adjustment: [0.42663577 0.84537478]
Processing patient index: 18  i is  5
Probability vector shape: (179,)

### turn ROI into habitats

In [25]:
### do it for each case - each patch

build = Build_list.Build(os.path.join('/host/d/Data/Habitats/Jishuitan/Patient_lists', 'labels_with_image_info_included.xlsx'))
batch_list, patient_index_list, label_list, image_path_list, mask_path_list = build.__build__()
print(f'Number of cases to process: {len(image_path_list)}')

for i in range(0, len(patient_index_list)):
    patient_index = patient_index_list[i]
    print('Processing patient index:', patient_index, ' i is ', i)

    # if done skip
    if os.path.isfile( os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'ROI_low.nii.gz')) == 1:
        print('done, continue')
        continue
    
    radiomics_path = os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'radiomics_features_patches_prob_clusters.xlsx')

    # load a template patch
    template_patch_file = os.path.join('/host/d/Data/Habitats/Jishuitan/habitats/', str(patient_index), 'patches/patch_0000.nii.gz')
    template_patch = nb.load(template_patch_file).get_fdata()
    affine = nb.load(template_patch_file).affine
    # turn it into all zeros
    ROI_high = np.zeros_like(template_patch)
    ROI_low = np.zeros_like(template_patch)


    cluster_info = pd.read_excel(radiomics_path)
    # find all the rows with cluster
    high_cluster_rows = cluster_info[cluster_info['cluster'] == 1]
    low_cluster_rows = cluster_info[cluster_info['cluster'] == 0]

    for j in range(0,high_cluster_rows.shape[0]):
        patch_id = high_cluster_rows.iloc[j]['patch_id']
        patch_file = os.path.join('/host/d/Data/Habitats/Jishuitan/habitats/', str(patient_index), f'patches/patch_{patch_id:04d}.nii.gz')
        patch_data = nb.load(patch_file).get_fdata()
        ROI_high += patch_data
    
    # assert ROI high only has 0 or 1 value
    assert np.all((ROI_high == 0) | (ROI_high == 1))

    for j in range(0,low_cluster_rows.shape[0]):
        patch_id = low_cluster_rows.iloc[j]['patch_id']
        patch_file = os.path.join('/host/d/Data/Habitats/Jishuitan/habitats/', str(patient_index), f'patches/patch_{patch_id:04d}.nii.gz')
        patch_data = nb.load(patch_file).get_fdata()
        ROI_low += patch_data
    assert np.all((ROI_low == 0) | (ROI_low == 1))


    # save 
    nb.save(nb.Nifti1Image(ROI_high, affine), os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'ROI_high.nii.gz'))
    nb.save(nb.Nifti1Image(ROI_low, affine), os.path.join('/host/d/projects/Habitats/radiomics/habitats', str(patient_index), 'ROI_low.nii.gz'))


Number of cases to process: 81
Processing patient index: 5  i is  0
done, continue
Processing patient index: 7  i is  1
Processing patient index: 8  i is  2
Processing patient index: 11  i is  3
Processing patient index: 15  i is  4
Processing patient index: 18  i is  5
Processing patient index: 19  i is  6
Processing patient index: 20  i is  7
Processing patient index: 21  i is  8
Processing patient index: 22  i is  9
Processing patient index: 23  i is  10
Processing patient index: 24  i is  11
Processing patient index: 26  i is  12
Processing patient index: 28  i is  13
Processing patient index: 29  i is  14
Processing patient index: 30  i is  15
Processing patient index: 33  i is  16
Processing patient index: 34  i is  17
Processing patient index: 36  i is  18
Processing patient index: 38  i is  19
Processing patient index: 41  i is  20
Processing patient index: 42  i is  21
Processing patient index: 43  i is  22
Processing patient index: 44  i is  23
Processing patient index: 46  i